# 19 — Compose manuscript figures

In [ ]:
from pathlib import Path
import sys, shutil

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D

from analysis.lib.plotting import (VARIABLE_LABELS, VARIABLE_LABELS_SHORT,
                                   PALETTE, apply_manuscript_style)
apply_manuscript_style()

CORE       = PROJECT_ROOT / 'outputs' / 'predictive_modeling_core' / 'figures'
POOLED_DIR = PROJECT_ROOT / 'outputs' / 'predictive_modeling_core' / 'pooled'
SS_CURVES  = PROJECT_ROOT / 'outputs' / 'predictive_modeling_core' / 'sample_size_curves' / 'figures'
SS_CSV     = PROJECT_ROOT / 'outputs' / 'predictive_modeling_core' / 'sample_size_curves' / 'sample_size_curves.csv'
SENS       = PROJECT_ROOT / 'outputs' / 'predictive_modeling_sensitivity' / 'figures'
BACKEND    = PROJECT_ROOT / 'outputs' / 'backend_comparison'
BRANCH     = PROJECT_ROOT / 'outputs' / 'predictive_modeling_branches' / 'figures'
EXP        = PROJECT_ROOT / 'outputs' / 'subject_level_analyses'
BETAS      = PROJECT_ROOT / 'outputs' / 'feature_importance' / 'ridge_betas'
FIG2_BRAINS_DIR = (PROJECT_ROOT / 'outputs' / 'feature_importance' / 'figures'
                   / 'cells_fig2_ridge_betas')
FIG3_CLUST_DIR  = (PROJECT_ROOT / 'outputs' / 'feature_importance' / 'figures'
                   / 'cells_fig2_clusters')
STRATA_FIG = PROJECT_ROOT / 'outputs' / 'predictive_modeling_volume_strata' / 'figures'

OUT_DIR  = PROJECT_ROOT / 'outputs' / 'manuscript_figures'
SUPP_DIR = PROJECT_ROOT / 'manuscript' / 'figures' / 'supplement'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUPP_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = ['cognition-PSCI_present',
           'cognition-aef_domain_score',
           'cognition-ps_domain_score',
           'cognition-language_domain_score',
           'cognition-verbalmemory_domain_score',
           'cognition-vsfunctions_domain_score',
           'cognition-vsmemory_domain_score']

CLIN_PATH = PROJECT_ROOT / "data" / "tabular" / "clinical_data.csv"


def imshow_png(ax, path):
    if not Path(path).exists():
        ax.text(0.5, 0.5, f'(missing)\n{Path(path).name}',
                ha='center', va='center', fontsize=10, color='gray',
                transform=ax.transAxes)
    else:
        ax.imshow(mpimg.imread(path))
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


def gray_divider(fig, y, x0=0.02, x1=0.98, color='#cccccc', lw=0.8):
    fig.add_artist(Line2D([x0, x1], [y, y], color=color, linewidth=lw,
                          transform=fig.transFigure, zorder=0))


Z_CUT = 2.0


def load_beta(target):
    df = pd.read_csv(BETAS / f'sLNM_{target}.csv', index_col=0)
    z = df['mean_beta'].abs() / df['se_beta'].replace(0, np.nan)
    df['z'] = z
    df['beta_thr'] = np.where(z >= Z_CUT, df['mean_beta'], np.nan)
    return df


def top_k_positive(df, k=10):
    s = df.dropna(subset=['beta_thr'])
    return s[s['mean_beta'] > 0].nlargest(k, 'mean_beta')


assert (BETAS / f"sLNM_{TARGETS[0]}.csv").exists(), \
    "run notebook 14 first (ridge-beta CSVs missing)"
all_betas = {t: load_beta(t) for t in TARGETS}
print('setup OK')

In [ ]:
# Figure 2 — performance (manuscript Fig 2; file main_fig1_performance)
DROP_VIEW_LABEL = {"lesion_acute": "Lesion location", "flnm_acute": "fLNM",
                   "slnm_acute": "sLNM"}
DROP_VIEW_COLOR = {"lesion_acute": PALETTE["Lesion"], "flnm_acute": PALETTE["fLNM"],
                   "slnm_acute": PALETTE["sLNM"]}
CLF_PALETTE = {"Ridge": "#9B7EBD", "XGBoost": "#6FB1A8", "TabPFN": "#E8A87C"}
ARM_LABELS = {"majority": "Majority", "permuted": "Permuted",
              "a": "Baseline", "b": "Integrated"}
ARM_COLORS = {"majority": PALETTE["Majority"], "permuted": PALETTE["Permuted"],
              "a": PALETTE["Baseline"], "b": PALETTE["Stacking"]}
ARM_LS = {"majority": ":", "permuted": "--", "a": "-", "b": "-"}


def render_panel_dropone(ax):
    dropone = pd.read_csv(POOLED_DIR / "stacking_dropone_scores.csv")
    stacker = pd.read_csv(POOLED_DIR / "stacking_scores.csv")
    full_auprc = stacker.set_index(["Target", "Fold"])["AUPRC"]
    rows = []
    for _, r in dropone.iterrows():
        full = full_auprc.get((r["Target"], r["Fold"]))
        if pd.notna(full) and pd.notna(r["AUPRC"]):
            rows.append({"Target": r["Target"], "DroppedView": r["DroppedView"],
                         "drop": float(full) - float(r["AUPRC"])})
    drop_df = pd.DataFrame(rows)
    summary = (drop_df.groupby(["Target", "DroppedView"])["drop"]
               .agg(["mean", "sem"]).reset_index())
    targets = [t for t in TARGETS if t in summary["Target"].unique()]
    views = ["lesion_acute", "flnm_acute", "slnm_acute"]
    n_t = len(targets); n_v = len(views)
    group_w = 0.85; band_w = group_w / n_v
    offsets = np.linspace(-group_w/2 + band_w/2, group_w/2 - band_w/2, n_v)
    for xi, tname in enumerate(targets):
        for gi, view in enumerate(views):
            row = summary[(summary["Target"] == tname)
                          & (summary["DroppedView"] == view)]
            if row.empty:
                continue
            m = float(row["mean"].iloc[0])
            s = float(row["sem"].iloc[0]) if pd.notna(row["sem"].iloc[0]) else 0
            cx = xi + offsets[gi]
            ax.bar(cx, m, width=band_w*0.95, color=DROP_VIEW_COLOR[view],
                   edgecolor="black", linewidth=0.5,
                   label=DROP_VIEW_LABEL[view] if xi == 0 else None)
            if s > 0:
                ax.errorbar(cx, m, yerr=s, color="black", lw=0.6, capsize=2, zorder=4)
    ax.axhline(0, color="black", lw=0.6)
    for x in range(1, n_t):
        ax.axvline(x - 0.5, color="gray", lw=0.3, alpha=0.4)
    ax.set_xticks(range(n_t))
    ax.set_xticklabels([VARIABLE_LABELS_SHORT[t] for t in targets],
                       rotation=35, ha="right", fontsize=13)
    ax.set_ylabel("Performance difference \n(ΔAUPRC, full − leave-one-set-out)",
                  fontsize=15, fontweight="bold")
    ax.legend(loc="lower center", bbox_to_anchor=(0.55, 1.02), ncol=3,
              frameon=False, fontsize=11, handletextpad=0.5,
              columnspacing=1.0, borderaxespad=0)


def render_panel_backend(ax):
    PATHS = {
        "Ridge":   POOLED_DIR / "stacking_scores.csv",
        "XGBoost": BACKEND / "xgboost" / "pooled" / "stacking_scores.csv",
        "TabPFN":  BACKEND / "tabpfn" / "pooled" / "stacking_scores.csv",
    }
    per_fold = {clf: pd.read_csv(p) for clf, p in PATHS.items() if p.exists()}
    ridge_auprc = per_fold["Ridge"].set_index(["Target", "Fold"])["AUPRC"]
    rows = []
    for clf in ("XGBoost", "TabPFN"):
        if clf not in per_fold:
            continue
        for _, r in per_fold[clf].iterrows():
            ref = ridge_auprc.get((r["Target"], r["Fold"]))
            if pd.notna(ref) and pd.notna(r["AUPRC"]):
                rows.append({"Target": r["Target"], "Classifier": clf,
                             "delta": float(r["AUPRC"]) - float(ref)})
    delta_df = pd.DataFrame(rows)
    # NB one-sided tests vs Ridge (notebook 04), FDR within comparison
    nb_bk = pd.read_csv(BACKEND / "nb_significance_pooled.csv")
    sig = dict(zip(zip(nb_bk["Comparison"], nb_bk["Target"]), nb_bk["p_adj_BH"]))
    summary = (delta_df.groupby(["Target", "Classifier"])["delta"]
               .agg(["mean", "sem"]).reset_index())
    targets = [t for t in TARGETS if t in summary["Target"].unique()]
    clfs = ["XGBoost", "TabPFN"]
    n_t = len(targets); n_c = len(clfs)
    group_w = 0.85; band_w = group_w / n_c
    offsets = np.linspace(-group_w/2 + band_w/2, group_w/2 - band_w/2, n_c)
    for xi, tname in enumerate(targets):
        for gi, clf in enumerate(clfs):
            row = summary[(summary["Target"] == tname)
                          & (summary["Classifier"] == clf)]
            if row.empty:
                continue
            m = float(row["mean"].iloc[0])
            s = float(row["sem"].iloc[0]) if pd.notna(row["sem"].iloc[0]) else 0
            cx = xi + offsets[gi]
            ax.bar(cx, m, width=band_w*0.95, color=CLF_PALETTE[clf],
                   edgecolor="black", linewidth=0.5,
                   label=f"{clf} − Ridge" if xi == 0 else None)
            if s > 0:
                ax.errorbar(cx, m, yerr=s, color="black", lw=0.6, capsize=2, zorder=4)
            if sig.get((f"{clf} > Ridge", tname), 1.0) < 0.05:
                ax.text(cx, m + s + 0.0015, "*", ha="center", va="bottom",
                        fontsize=16, fontweight="bold")
    ax.axhline(0, color="black", lw=0.6)
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi + 0.15 * (hi - lo))   # headroom for significance stars
    for x in range(1, n_t):
        ax.axvline(x - 0.5, color="gray", lw=0.3, alpha=0.4)
    ax.set_xticks(range(n_t))
    ax.set_xticklabels([VARIABLE_LABELS_SHORT[t] for t in targets],
                       rotation=35, ha="right", fontsize=13)
    ax.set_ylabel("Performance difference (ΔAUPRC)", fontsize=15, fontweight="bold")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2,
              frameon=False, fontsize=11, handletextpad=0.5,
              columnspacing=1.0, borderaxespad=0)


def render_panel_samplesize(ax):
    ss = pd.read_csv(SS_CSV)
    psci = ss[ss["Target"] == "cognition-PSCI_present"]
    _summ = pd.read_csv(POOLED_DIR / "modeling_summary.csv")
    full_n = int(_summ.loc[_summ["Target"] == "cognition-PSCI_present", "N"].iloc[0])
    train_n = full_n * 0.9
    for arm in ["majority", "permuted", "a", "b"]:
        a_sub = psci[psci["arm"] == arm]
        if a_sub.empty:
            continue
        fold_means = a_sub.groupby(["frac", "fold"])["AUPRC"].mean().reset_index()
        agg = fold_means.groupby("frac")["AUPRC"].agg(["mean", "std", "count"]).reset_index()
        agg["se"] = agg["std"] / np.sqrt(agg["count"].clip(lower=1))
        xs = agg["frac"].values * train_n
        ax.plot(xs, agg["mean"], color=ARM_COLORS[arm], lw=2.2, ls=ARM_LS[arm],
                marker="o", markersize=6, label=ARM_LABELS[arm])
        ax.fill_between(xs, agg["mean"] - agg["se"], agg["mean"] + agg["se"],
                        color=ARM_COLORS[arm], alpha=0.18)
    ax.set_xlabel("N$_{train}$", fontsize=15)
    ax.set_ylabel("Predictive performance (AUPRC)", fontsize=15)
    ax.set_ylim(0.30, 0.85)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2,
              frameon=False, fontsize=11, handletextpad=0.5,
              columnspacing=1.5, borderaxespad=0)


fig = plt.figure(figsize=(20, 22))
outer = GridSpec(3, 1, figure=fig, hspace=0.26,
                 left=0.045, right=0.955, top=0.985, bottom=0.025,
                 height_ratios=[1.0, 0.55, 1.0])

axA = fig.add_subplot(outer[0])
imshow_png(axA, CORE / "pooled_cv_violin.png")
axA.set_title("a. Predictive performance by cognitive target", loc="left", fontsize=14)

sub_bcd = GridSpecFromSubplotSpec(1, 3, subplot_spec=outer[1], wspace=0.40,
                                  width_ratios=[1.0, 1.0, 1.0])
axB = fig.add_subplot(sub_bcd[0]); render_panel_dropone(axB)
axB.set_title("b. Feature set ablation", loc="left", fontsize=14, pad=30)
axC = fig.add_subplot(sub_bcd[1]); render_panel_samplesize(axC)
axC.set_title("c. Sample-size curves (any-domain PSCI)", loc="left", fontsize=14, pad=30)
axD = fig.add_subplot(sub_bcd[2]); render_panel_backend(axD)
axD.set_title("d. Algorithm comparison", loc="left", fontsize=14, pad=30)

_pos_b = axB.get_position()
axB.set_position([_pos_b.x0 + 0.06, _pos_b.y0 + 0.02, _pos_b.width, _pos_b.height])
_pos_c = axC.get_position()
axC.set_position([_pos_c.x0 + 0.035, _pos_c.y0 + 0.02, _pos_c.width, _pos_c.height])
_pos_d = axD.get_position()
axD.set_position([_pos_d.x0 + 0.02, _pos_d.y0 + 0.02, _pos_d.width, _pos_d.height])

axE = fig.add_subplot(outer[2])
imshow_png(axE, STRATA_FIG / "volume_strata_delta_strip.png")
axE.set_title("e. Performance difference between integrated and baseline models, "
              "stratified by infarct volume", loc="left", fontsize=14)

gray_divider(fig, outer[0].get_position(fig).y0 - 0.005, x0=0.10, x1=0.90)
gray_divider(fig, outer[2].get_position(fig).y1 + 0.01, x0=0.10, x1=0.90)

fig.savefig(OUT_DIR / "main_fig1_performance.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig1_performance.svg", bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig1_performance.pdf", bbox_inches="tight")
print("saved", OUT_DIR / "main_fig1_performance.png")

In [ ]:
# Figure 3 — per-target Ridge β on the brain (file main_fig2_ridge_betas)
_NETTEKOVEN_CEREBELLUM = {
    "M1", "M2", "M3", "M4", "A1", "A2", "A3",
    "D1", "D2", "D3", "D4", "S1", "S2", "S3", "S4", "S5",
}
_TIAN_SUBCORTEX = {
    "HIP": "Hippocampus", "AMY": "Amygdala", "pTHA": "Posterior thalamus",
    "aTHA": "Anterior thalamus", "NAc": "Nucleus accumbens",
    "GP": "Globus pallidus", "PUT": "Putamen", "CAU": "Caudate",
}
_SCHAEFER_NETWORKS = {
    "Vis": "Visual", "SomMot": "Somatomotor", "DorsAttn": "Dorsal attention",
    "SalVentAttn": "Salience/ventral attention", "Limbic": "Limbic",
    "Cont": "Control", "Default": "Default mode",
}
_SCHAEFER_REGIONS = {
    "PFC": "PFC", "PFCv": "ventral PFC", "TempPole": "temporal pole",
    "PrCv": "precentral ventral", "Post": "posterior",
    "pCunPCC": "precuneus/PCC", "SomMot": "somatomotor",
}


def display_parcel(parcel: str) -> str:
    if len(parcel) >= 2 and parcel[-1] in ("L", "R") and parcel[:-1] in _NETTEKOVEN_CEREBELLUM:
        hemi = "left" if parcel[-1] == "L" else "right"
        return f"Cerebellum {parcel[:-1]} {hemi}"
    if parcel.endswith("-lh") or parcel.endswith("-rh"):
        hemi = "left" if parcel.endswith("-lh") else "right"
        return f"{_TIAN_SUBCORTEX.get(parcel[:-3], parcel[:-3])} {hemi}"
    if parcel.startswith("7Networks_"):
        parts = parcel.split("_")
        hemi = "left" if parts[1] == "LH" else "right"
        net = _SCHAEFER_NETWORKS.get(parts[2], parts[2])
        idx = parts[-1]
        if len(parts) > 4:
            region_raw = "_".join(parts[3:-1])
            region = _SCHAEFER_REGIONS.get(region_raw, region_raw)
            return f"{net} {region} {hemi} {idx}"
        return f"{net} {hemi} {idx}"
    if parcel.endswith(" L") or parcel.endswith(" R"):
        hemi = "left" if parcel.endswith(" L") else "right"
        base = parcel[:-2]
        return f"{base[0].upper()}{base[1:].lower()} {hemi}"
    return parcel


def _render_brain_block(fig, subspec, betas_by_target=None, brains_dir=None):
    betas_by_target = betas_by_target if betas_by_target is not None else all_betas
    brains_dir = brains_dir or FIG2_BRAINS_DIR
    WR = [2.333, 1.400, 1.000, 1.000]
    LABEL_W = 2.55
    SECTION_TITLES = ["Cortex (Schaefer100)", "Subcortex (Tian)",
                      "White matter (XTRACT)", "Cerebellum (Nettekoven)"]
    SECTIONS = ["cortex", "subcortex", "tracts", "cerebellum"]
    TARGETS_SHORT = [
        ("PSCI",     "cognition-PSCI_present"),
        ("aef",      "cognition-aef_domain_score"),
        ("ps",       "cognition-ps_domain_score"),
        ("language", "cognition-language_domain_score"),
        ("verbmem",  "cognition-verbalmemory_domain_score"),
        ("vsfunc",   "cognition-vsfunctions_domain_score"),
        ("vsmem",    "cognition-vsmemory_domain_score"),
    ]
    sub = GridSpecFromSubplotSpec(
        8, 5, subplot_spec=subspec, wspace=0.06, hspace=0.02,
        width_ratios=[LABEL_W] + WR, height_ratios=[1.0] * 7 + [0.10])
    for r, (short, full) in enumerate(TARGETS_SHORT):
        ax_lab = fig.add_subplot(sub[r, 0])
        ax_lab.set_xticks([]); ax_lab.set_yticks([])
        for sp in ax_lab.spines.values():
            sp.set_visible(False)
        ax_lab.text(0.02, 0.97, VARIABLE_LABELS[full],
                    ha="left", va="top", fontsize=13, fontweight="bold",
                    transform=ax_lab.transAxes)
        top5 = top_k_positive(betas_by_target[full], k=5)
        if len(top5):
            roi_lines = "\n".join(
                f"{display_parcel(roi):<42s}  β = {float(row.mean_beta):+.3f}"
                for roi, row in top5.iterrows())
            ax_lab.text(0.02, 0.70, roi_lines, ha="left", va="top",
                        fontsize=12, fontweight="bold",
                        family="DejaVu Sans Mono", transform=ax_lab.transAxes)
        for cc, section in enumerate(SECTIONS):
            ax = fig.add_subplot(sub[r, cc + 1])
            png = brains_dir / f"{short}_{section}.png"
            if png.exists():
                ax.imshow(mpimg.imread(png))
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)
            if r == 0:
                ax.set_title(SECTION_TITLES[cc], fontsize=12,
                             fontweight="bold", pad=4, y=1.0)
    shared_vmax = float(pd.read_csv(brains_dir / "shared_vmax.csv")["vmax"].iloc[0])
    norm = plt.Normalize(-shared_vmax, shared_vmax)
    bb_l = sub[7, 1].get_position(fig)
    bb_r = sub[7, 4].get_position(fig)
    cb_w = (bb_r.x1 - bb_l.x0) * 0.55
    cb_x = bb_l.x0 + ((bb_r.x1 - bb_l.x0) - cb_w) / 2
    cb_ax = fig.add_axes([cb_x, bb_l.y0, cb_w, 0.010])
    sm_cb = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
    sm_cb.set_array([])
    cb = fig.colorbar(sm_cb, cax=cb_ax, orientation="horizontal")
    cb.set_label("Ridge β", fontsize=12, fontweight="bold")
    cb.ax.tick_params(labelsize=9)
    cb.outline.set_linewidth(0.3)


fig = plt.figure(figsize=(20, 19))
outer = GridSpec(1, 1, figure=fig,
                 left=0.04, right=0.985, top=0.965, bottom=0.025)
_render_brain_block(fig, outer[0])
fig.savefig(OUT_DIR / "main_fig2_ridge_betas.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig2_ridge_betas.svg", bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig2_ridge_betas.pdf", bbox_inches="tight")
print("saved", OUT_DIR / "main_fig2_ridge_betas.png")

In [ ]:
# Figs S17, S18: coefficient maps for lesion location and functional LNM
# (same layout as Fig 3; per-modality shared vmax from notebook 14)
SUPPL_BETA_MODS = {
    "flnm":   ("fLNM",   PROJECT_ROOT / "outputs" / "feature_importance"
               / "figures" / "cells_suppl_betas_flnm"),
    "lesion": ("Lesion", PROJECT_ROOT / "outputs" / "feature_importance"
               / "figures" / "cells_suppl_betas_lesion"),
}
for tag, (modality, brains_dir) in SUPPL_BETA_MODS.items():
    if not (brains_dir / "shared_vmax.csv").exists():
        print(f"[skip suppl betas {tag}: run notebook 14 first]")
        continue
    mod_betas = {}
    for tgt in TARGETS:
        df = pd.read_csv(BETAS / f"{modality}_{tgt}.csv", index_col=0)
        z = df["mean_beta"].abs() / df["se_beta"].replace(0, np.nan)
        df["z"] = z
        df["beta_thr"] = np.where(z >= Z_CUT, df["mean_beta"], np.nan)
        mod_betas[tgt] = df
    fig = plt.figure(figsize=(20, 19))
    outer = GridSpec(1, 1, figure=fig,
                     left=0.04, right=0.985, top=0.965, bottom=0.025)
    _render_brain_block(fig, outer[0], betas_by_target=mod_betas,
                        brains_dir=brains_dir)
    out_png = OUT_DIR / f"suppl_ridge_betas_{tag}.png"
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"saved {out_png}")

In [ ]:
# Fig S16: top-region × domain overlap heatmap (sLNM). Rows = union of each
# domain's top-5 positive z-thresholded β regions; cells = mean β; black boxes
# mark top-5 membership. Rows ordered by number of domains hit.
import seaborn as sns

K_TOP = 5
top_sets = {t: list(top_k_positive(all_betas[t], k=K_TOP).index) for t in TARGETS}
union_regions = []
for t in TARGETS:
    for p in top_sets[t]:
        if p not in union_regions:
            union_regions.append(p)

M = pd.DataFrame({t: all_betas[t]["mean_beta"].reindex(union_regions)
                  for t in TARGETS})
in_top = pd.DataFrame({t: [p in top_sets[t] for p in union_regions]
                       for t in TARGETS}, index=union_regions)
row_order = in_top.sum(axis=1).sort_values(ascending=False, kind="stable").index
M = M.loc[row_order]
in_top = in_top.loc[row_order]
M.to_csv(OUT_DIR / "suppl_beta_top_regions_matrix.csv")

vmax_hm = float(np.nanmax(np.abs(M.values)))
fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(union_regions) + 2.2))
sns.heatmap(M, cmap="RdBu_r", center=0, vmin=-vmax_hm, vmax=vmax_hm,
            annot=True, fmt=".3f", annot_kws={"fontsize": 8},
            linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Ridge β (sLNM)", "shrink": 0.6}, ax=ax)
for yi, p in enumerate(M.index):
    for xi, t in enumerate(TARGETS):
        if in_top.loc[p, t]:
            ax.add_patch(plt.Rectangle((xi, yi), 1, 1, fill=False,
                                       edgecolor="black", lw=1.8))
ax.set_xticklabels([VARIABLE_LABELS_SHORT[t] for t in TARGETS],
                   rotation=45, ha="right", fontsize=10)
ax.set_yticklabels([display_parcel(p) for p in M.index], rotation=0, fontsize=9)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
out_png = OUT_DIR / "suppl_beta_top_regions_heatmap.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print(f"saved {out_png}")

In [ ]:
# Figure 4 — sLNM UMAP story (file main_fig3_clusters)
UMAP_CSV  = EXP / "umap_sLNM_subjects.csv"
PRED_ALL7 = EXP / "pooled_all7_per_subject_predictions.csv"

DOMAINS_FIG3 = [
    ("cognition-aef_domain_score",          "Attn / Exec"),
    ("cognition-ps_domain_score",           "Processing speed"),
    ("cognition-language_domain_score",     "Language"),
    ("cognition-verbalmemory_domain_score", "Verbal memory"),
    ("cognition-vsfunctions_domain_score",  "Visuospatial functions"),
    ("cognition-vsmemory_domain_score",     "Visuospatial memory"),
]

# Cluster names derived from per-cluster mean sLNM peak voxel coordinates
# (9-cluster HDBSCAN, min_cluster_size=50, min_samples=10).
CLUSTER_NAMES = {
    0: "L PCA / AChA",
    1: "ACA",
    2: "Infratentorial",
    3: "L deep perforator",
    4: "L MCA",
    5: "R deep perforator",
    6: "R MCA",
    7: "R PCA / AChA",
    8: "Diverse",
}
CLUSTER_LABEL_NUDGE = {0: (-2.6, 0.0), 1: (+1.2, 0.0), 6: (+1.0, 0.0)}

IMPAIR_COLOR_POS = "#c62828"
IMPAIR_COLOR_NEG = "#5e8eb6"
IMPAIR_COLOR_NA  = (0.90, 0.90, 0.90, 0.45)


def _strip_axes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")
    for sp in ax.spines.values():
        sp.set_visible(False)


def _scatter_binary(ax, df, y_col, *, dot_sizes=(6, 10, 12)):
    m_na = df[y_col].isna()
    m_neg = (df[y_col] == 0)
    m_pos = (df[y_col] == 1)
    s_na, s_neg, s_pos = dot_sizes
    if m_na.any():
        ax.scatter(df.loc[m_na, "umap_x"], df.loc[m_na, "umap_y"],
                   s=s_na, c=[IMPAIR_COLOR_NA], linewidths=0)
    if m_neg.any():
        ax.scatter(df.loc[m_neg, "umap_x"], df.loc[m_neg, "umap_y"],
                   s=s_neg, c=IMPAIR_COLOR_NEG, alpha=0.75, linewidths=0)
    if m_pos.any():
        ax.scatter(df.loc[m_pos, "umap_x"], df.loc[m_pos, "umap_y"],
                   s=s_pos, c=IMPAIR_COLOR_POS, alpha=0.85, linewidths=0)
    return int((~m_na).sum()), (float(df.loc[~m_na, y_col].mean())
                                if (~m_na).any() else float("nan"))


def _binary_legend_handles():
    return [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=IMPAIR_COLOR_POS,
               markersize=9, markeredgewidth=0, label="impaired"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=IMPAIR_COLOR_NEG,
               markersize=9, markeredgewidth=0, label="not impaired"),
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor=IMPAIR_COLOR_NA[:3] + (1.0,),
               markersize=9, markeredgewidth=0, label="no label"),
    ]


def _render_top_row(fig, subspec):
    sub = GridSpecFromSubplotSpec(1, 3, subplot_spec=subspec, wspace=0.10)
    df = pd.read_csv(UMAP_CSV)
    pred = pd.read_csv(PRED_ALL7)
    pred_psci = pred[pred["target"] == "cognition-PSCI_present"][
        ["subject_id", "y_true", "mean_meta_proba"]]
    pred_psci = pred_psci.rename(columns={"mean_meta_proba": "psci_prob",
                                          "y_true": "psci_true"})
    df = df.merge(pred_psci, on="subject_id", how="left")

    clusters = sorted([c for c in df["cluster"].unique() if c >= 0])
    cmap_cl = plt.cm.tab10
    colors = {c: cmap_cl(i % 10) for i, c in enumerate(clusters)}
    colors[-1] = (0.75, 0.75, 0.75, 0.55)

    ax_a = fig.add_subplot(sub[0, 0])
    noise = df[df["cluster"] == -1]
    ax_a.scatter(noise["umap_x"], noise["umap_y"], s=10, c=[colors[-1]],
                 alpha=0.5, linewidths=0)
    for c in clusters:
        sub_c = df[df["cluster"] == c]
        ax_a.scatter(sub_c["umap_x"], sub_c["umap_y"], s=18, c=[colors[c]],
                     alpha=0.78, linewidths=0)
        cx, cy = sub_c["umap_x"].mean(), sub_c["umap_y"].mean()
        dx, dy = CLUSTER_LABEL_NUDGE.get(c, (0.0, 0.0))
        rate = (sub_c["psci_true"].dropna().mean()
                if sub_c["psci_true"].notna().any() else None)
        name = CLUSTER_NAMES.get(c, f"Cluster {c}")
        disp = c + 1
        lab = f"{disp}: {name}\n{rate*100:.0f}%" if rate is not None else f"{disp}: {name}"
        ax_a.text(cx + dx, cy + dy, lab, fontsize=10, fontweight="bold",
                  color="black", ha="center", va="center",
                  bbox=dict(boxstyle="round,pad=0.28", fc="white",
                            ec="black", lw=1.0, alpha=0.92), zorder=10)
    ax_a.set_title("a. UMAP of structural LNM features",
                   loc="left", fontsize=14, fontweight="bold")
    ax_a.text(0.1, 0.08, "Coloured by HDBSCAN cluster", fontsize=12,
              transform=ax_a.transAxes)
    _strip_axes(ax_a)

    ax_b = fig.add_subplot(sub[0, 1])
    has_pred = df["psci_prob"].notna()
    no_pred = df[~has_pred]
    if len(no_pred):
        ax_b.scatter(no_pred["umap_x"], no_pred["umap_y"], s=10,
                     c=[(0.85, 0.85, 0.85, 0.5)], linewidths=0)
    vmin = float(df.loc[has_pred, "psci_prob"].quantile(0.05))
    vmax = float(df.loc[has_pred, "psci_prob"].quantile(0.95))
    sc = ax_b.scatter(df.loc[has_pred, "umap_x"], df.loc[has_pred, "umap_y"],
                      s=18, c=df.loc[has_pred, "psci_prob"], cmap="plasma",
                      vmin=vmin, vmax=vmax, alpha=0.88, linewidths=0)
    cb = plt.colorbar(sc, ax=ax_b, shrink=0.55, pad=0.01, location="right")
    cb.set_label(f"PSCI probability\n(clip {vmin:.2f}\u2013{vmax:.2f})",
                 fontsize=11, fontweight="bold")
    cb.ax.tick_params(labelsize=10)
    ax_b.set_title("b. Predicted PSCI probability (integrated model)",
                   loc="left", fontsize=14, fontweight="bold")
    _strip_axes(ax_b)

    ax_c = fig.add_subplot(sub[0, 2])
    _scatter_binary(ax_c, df, "psci_true", dot_sizes=(10, 16, 18))
    ax_c.set_title("c. Ground-truth PSCI",
                   loc="left", fontsize=14, fontweight="bold", x=-0.025)
    ax_c.legend(handles=_binary_legend_handles()[:2], loc="upper right",
                fontsize=10, frameon=False, handletextpad=0.4, labelspacing=0.6)
    _strip_axes(ax_c)


def _render_cluster_slices(fig, subspec):
    stats = pd.read_csv(EXP / "umap_sLNM_cluster_stats.csv")
    cluster_ids = sorted([int(c) for c in stats["cluster"].unique() if c >= 0])
    if len(cluster_ids) == 9:
        n_cols, n_rows = 3, 3
    else:
        n_cols = 2
        n_rows = (len(cluster_ids) + n_cols - 1) // n_cols
    sub = GridSpecFromSubplotSpec(n_rows, n_cols + 1, subplot_spec=subspec,
                                  wspace=0.02, hspace=0.02,
                                  width_ratios=[1.0] * n_cols + [0.025])
    panel_axes = []
    for i, cid in enumerate(cluster_ids):
        r, c = divmod(i, n_cols)
        ax = fig.add_subplot(sub[r, c])
        png = FIG3_CLUST_DIR / f"cluster_+{cid}_axial.png"
        if png.exists():
            ax.imshow(mpimg.imread(png))
        row = stats[stats["cluster"] == cid]
        n_sub = int(row["n"].iloc[0]) if len(row) else 0
        name = CLUSTER_NAMES.get(cid, f"Cluster {cid}")
        ax.set_title(f"{cid + 1}: {name}  (N = {n_sub})",
                     fontsize=14, fontweight="bold")
        _strip_axes(ax)
        panel_axes.append(ax)
    mid_row = n_rows // 2
    cbax = fig.add_subplot(sub[mid_row, n_cols])
    _pos = cbax.get_position()
    cbax.set_position([_pos.x0 + 0.025, _pos.y0, _pos.width, _pos.height])
    cbax.imshow(np.linspace(0, 1, 256).reshape(-1, 1),
                cmap="hot", aspect="auto", origin="lower")
    cbax.set_xticks([])
    # 0% and the 95th-percentile shared vmax from the cluster-slice renderer.
    cbax.set_yticks([0, 255])
    cbax.set_yticklabels(["0", "28"], fontsize=11, fontweight="bold")
    for sp in cbax.spines.values():
        sp.set_linewidth(0.5)
    cbax.set_title("%", fontsize=12, fontweight="bold", pad=6)
    return panel_axes


def _add_radiological_overlay(fig, panel_axes):
    if not panel_axes:
        return
    boxes = [ax.get_position() for ax in panel_axes]
    y_mid = 0.5 * (max(b.y1 for b in boxes) + min(b.y0 for b in boxes))
    x_left = min(b.x0 for b in boxes) - 0.018
    x_right = max(b.x1 for b in boxes) + 0.005
    txt_kw = dict(fontsize=22, fontweight="bold", color="#222",
                  ha="center", va="center")
    fig.text(x_left, y_mid, "R", **txt_kw)
    fig.text(x_right, y_mid, "L", **txt_kw)


def _render_domain_prob_row(fig, subspec):
    n = len(DOMAINS_FIG3)
    outer_row = GridSpecFromSubplotSpec(2, 1, subplot_spec=subspec, hspace=0.65,
                                        height_ratios=[0.10, 1.0])
    ax_title = fig.add_subplot(outer_row[0])
    ax_title.set_xticks([]); ax_title.set_yticks([])
    for sp in ax_title.spines.values():
        sp.set_visible(False)
    ax_title.text(0.0, 0.5,
                  "e. Predicted probability for each cognitive domain (integrated model)",
                  ha="left", va="center", fontsize=15, fontweight="bold",
                  transform=ax_title.transAxes)
    sub = GridSpecFromSubplotSpec(1, n + 1, subplot_spec=outer_row[1],
                                  wspace=0.10, width_ratios=[1.0] * n + [0.06])
    df_um = pd.read_csv(UMAP_CSV)[["subject_id", "umap_x", "umap_y"]]
    pred = pd.read_csv(PRED_ALL7)
    for i, (tgt, nice) in enumerate(DOMAINS_FIG3):
        ax = fig.add_subplot(sub[0, i])
        p_tgt = pred[pred["target"] == tgt][["subject_id", "mean_meta_proba"]]
        merged = df_um.merge(p_tgt, on="subject_id", how="left")
        has = merged["mean_meta_proba"].notna()
        no = merged[~has]
        if len(no):
            ax.scatter(no["umap_x"], no["umap_y"], s=6,
                       c=[(0.88, 0.88, 0.88, 0.4)], linewidths=0)
        v_lo = float(merged.loc[has, "mean_meta_proba"].quantile(0.05))
        v_hi = float(merged.loc[has, "mean_meta_proba"].quantile(0.95))
        ax.scatter(merged.loc[has, "umap_x"], merged.loc[has, "umap_y"],
                   s=10, c=merged.loc[has, "mean_meta_proba"], cmap="plasma",
                   vmin=v_lo, vmax=v_hi, alpha=0.85, linewidths=0)
        ax.set_title(f"{nice}\nclip: {v_lo:.2f}\u2013{v_hi:.2f}",
                     fontsize=14, fontweight="bold")
        _strip_axes(ax)
    cbax = fig.add_subplot(sub[0, n])
    cbax.imshow(np.linspace(0, 1, 256).reshape(-1, 1),
                cmap="plasma", aspect="auto", origin="lower")
    cbax.set_xticks([])
    cbax.set_yticks([0, 255])
    _pos = cbax.get_position()
    cbax.set_position([_pos.x0 + 0.015, _pos.y0, _pos.width, _pos.height])
    cbax.set_yticklabels(["low", "high"], fontsize=12, fontweight="bold")
    for sp in cbax.spines.values():
        sp.set_linewidth(0.5)
    cbax.set_title("rel.\nprob.", fontsize=12, fontweight="bold", pad=13)


def _render_domain_binary_row(fig, subspec):
    n = len(DOMAINS_FIG3)
    outer_row = GridSpecFromSubplotSpec(2, 1, subplot_spec=subspec, hspace=0.65,
                                        height_ratios=[0.10, 1.0])
    ax_title = fig.add_subplot(outer_row[0])
    ax_title.set_xticks([]); ax_title.set_yticks([])
    for sp in ax_title.spines.values():
        sp.set_visible(False)
    ax_title.text(0.0, 0.5, "f. Ground-truth per cognitive domain",
                  ha="left", va="center", fontsize=15, fontweight="bold",
                  transform=ax_title.transAxes)
    sub = GridSpecFromSubplotSpec(1, n + 1, subplot_spec=outer_row[1],
                                  wspace=0.10, width_ratios=[1.0] * n + [0.06])
    df_um = pd.read_csv(UMAP_CSV)[["subject_id", "umap_x", "umap_y"]]
    pred = pd.read_csv(PRED_ALL7)
    for i, (tgt, nice) in enumerate(DOMAINS_FIG3):
        ax = fig.add_subplot(sub[0, i])
        p_tgt = pred[pred["target"] == tgt][["subject_id", "y_true"]]
        merged = df_um.merge(p_tgt, on="subject_id", how="left")
        _scatter_binary(ax, merged, "y_true")
        ax.set_title(f"{nice}", fontsize=14, fontweight="bold")
        _strip_axes(ax)
    leg_ax = fig.add_subplot(sub[0, n])
    leg_ax.set_xticks([]); leg_ax.set_yticks([])
    for sp in leg_ax.spines.values():
        sp.set_visible(False)
    leg_ax.legend(handles=_binary_legend_handles(), loc="center left",
                  fontsize=11, frameon=False,
                  handletextpad=0.4, labelspacing=1.1)


fig = plt.figure(figsize=(22, 34))
outer = GridSpec(4, 1, figure=fig, hspace=0.10,
                 left=0.04, right=0.985, top=0.98, bottom=0.020,
                 height_ratios=[0.70, 1.50, 0.50, 0.50])
_render_top_row(fig, outer[0])
sub_d_outer = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[1],
                                      hspace=0.03, height_ratios=[0.04, 1.0])
ax_d_title = fig.add_subplot(sub_d_outer[0])
ax_d_title.set_xticks([]); ax_d_title.set_yticks([])
for sp in ax_d_title.spines.values():
    sp.set_visible(False)
ax_d_title.text(0.0, 0.5,
                "d. Per-cluster mean voxel-level percentage of lesion-connected streamlines",
                ha="left", va="center", fontsize=15, fontweight="bold",
                transform=ax_d_title.transAxes)
slice_axes = _render_cluster_slices(fig, sub_d_outer[1])
_render_domain_prob_row(fig, outer[2])
_render_domain_binary_row(fig, outer[3])
_add_radiological_overlay(fig, slice_axes)

gray_divider(fig, outer[0].get_position(fig).y0 - 0.003, x0=0.10, x1=0.90)
gray_divider(fig, outer[1].get_position(fig).y0 - 0.003, x0=0.10, x1=0.90)
gray_divider(fig, outer[2].get_position(fig).y0 - 0.003, x0=0.10, x1=0.90)

fig.savefig(OUT_DIR / "main_fig3_clusters.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig3_clusters.svg", bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig3_clusters.pdf", bbox_inches="tight")
print("saved", OUT_DIR / "main_fig3_clusters.png")

In [ ]:
# Figure 5 — pre-existing cerebrovascular lesions (file main_fig4_chronic_acute)
BRANCH_FIG  = BRANCH / "branch_deltas_main_wmh_chronic.png"
PSCI_HEAT   = EXP / "figures" / "fig4_heatmap_psci.png"
DOMAIN_HEAT = EXP / "figures" / "fig4_heatmap_domains.png"
WMH_VT_FIG  = (PROJECT_ROOT / "outputs" / "predictive_modeling_wmh"
               / "volume_vs_topology" / "figures")

fig = plt.figure(figsize=(22, 22))
outer = GridSpec(3, 1, figure=fig, hspace=0.06,
                 left=0.04, right=0.985, top=0.96, bottom=0.025,
                 height_ratios=[1.05, 0.76, 1.16])

sub_top = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[0], wspace=0.06,
                                  width_ratios=[2.8, 1.0])
axA = fig.add_subplot(sub_top[0])
imshow_png(axA, BRANCH_FIG)
axA.set_anchor("N")

axA_psci = fig.add_subplot(sub_top[1])
imshow_png(axA_psci, PSCI_HEAT)
axA_psci.set_anchor("C")
_pos_pj = axA_psci.get_position()
axA_psci.set_position([_pos_pj.x0 - 0.025, _pos_pj.y0,
                       _pos_pj.width + 0.04, _pos_pj.height])

fig.canvas.draw()
row1_top = max(axA.get_position().y1, axA_psci.get_position().y1)
title_y = row1_top + 0.001
fig.text(axA.get_position().x0, title_y,
         "a. Predictive gain from adding pre-existing lesion topography and connectivity features",
         ha="left", va="bottom", fontsize=15, fontweight="bold")
fig.text(axA_psci.get_position().x0, title_y,
         "b. Any-domain PSCI rate by structural lesion connectivity",
         ha="left", va="bottom", fontsize=15, fontweight="bold")

axB = fig.add_subplot(outer[1])
imshow_png(axB, DOMAIN_HEAT)
axB.set_title("c. Impairment rate by structural lesion connectivity for cognitive domains",
              loc="left", fontsize=14, fontweight="bold")

gray_divider(fig, outer[0].get_position(fig).y0 - 0.011, x0=0.10, x1=0.65)
gray_divider(fig, outer[1].get_position(fig).y0 + 0.011, x0=0.10, x1=0.90)

axD = fig.add_subplot(outer[2])
imshow_png(axD, WMH_VT_FIG / "wmh_volume_vs_topology_violin.png")
axD.set_title("d. WMH topography and connectivity vs. WMH volume",
              loc="left", fontsize=14, fontweight="bold")

fig.savefig(OUT_DIR / "main_fig4_chronic_acute.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig4_chronic_acute.svg", bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig4_chronic_acute.pdf", bbox_inches="tight")
print("saved", OUT_DIR / "main_fig4_chronic_acute.png")

In [ ]:
# Figure S6 — LOCO cross-cohort story (file main_fig5_loco)
EXP_FIG = EXP / "figures"

fig = plt.figure(figsize=(18, 22))
outer = GridSpec(3, 1, figure=fig, hspace=0.08,
                 left=0.035, right=0.985, top=0.985, bottom=0.02,
                 height_ratios=[1.0, 0.60, 0.55])

axA = fig.add_subplot(outer[0])
imshow_png(axA, CORE / "loco_cv_panel_normauprc.png")
axA.set_anchor("S")
fig.text(0.035, axA.get_position().y1 + 0.02,
         "a. Cross-cohort generalization — out-of-sample normalised AUPRC "
         "by cognitive target (leave-one-cohort-out)",
         ha="left", va="bottom", fontsize=15, fontweight="bold")

axB = fig.add_subplot(outer[1])
imshow_png(axB, EXP_FIG / "loco_imaging_transfers.png")
axB.set_anchor("N")
fig.text(0.035, axB.get_position().y1 + 0.005,
         "b. Imaging contribution to leave-one-cohort-out any-domain PSCI "
         "prediction (integrated vs. baseline)",
         ha="left", va="bottom", fontsize=15, fontweight="bold")

sub_cd = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[2], wspace=0.06,
                                 width_ratios=[1.25, 1.0])
axC = fig.add_subplot(sub_cd[0])
imshow_png(axC, EXP_FIG / "loco_cohort_differences.png")
axC.set_anchor("N")
fig.text(0.035, axC.get_position().y1 + 0.005,
         "c. Systematic differences between cohorts",
         ha="left", va="bottom", fontsize=15, fontweight="bold")
axD = fig.add_subplot(sub_cd[1])
imshow_png(axD, EXP_FIG / "loco_drivers_concise.png")
axD.set_anchor("N")
fig.text(axD.get_position().x0, axD.get_position().y1 + 0.005,
         "d. Cohort-level factors associated with\n"
         "cross-cohort predictive performance",
         ha="left", va="bottom", fontsize=15, fontweight="bold")

_y_div_ab = (outer[0].get_position(fig).y0 + outer[1].get_position(fig).y1) / 2
gray_divider(fig, _y_div_ab + 0.008, x0=0.10, x1=0.90)
# divider must clear the two-line panel-d title above row 3
_y_div_bc = (outer[1].get_position(fig).y0 + outer[2].get_position(fig).y1) / 2
_title_top_cd = max(axC.get_position().y1, axD.get_position().y1) + 0.036
gray_divider(fig, max(_y_div_bc + 0.008, _title_top_cd), x0=0.10, x1=0.90)

fig.savefig(OUT_DIR / "main_fig5_loco.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig5_loco.svg", bbox_inches="tight")
fig.savefig(OUT_DIR / "main_fig5_loco.pdf", bbox_inches="tight")
print("saved", OUT_DIR / "main_fig5_loco.png")

In [ ]:
# Figures S9 / S10 — lesion-location and fLNM versions of the UMAP figure
SHORT_LABEL = {"lesion": "lesion-location", "flnm": "functional LNM"}
CLUSTER_LABEL_NUDGE_V = {
    "lesion": {9: (-0.6, 0.0), 12: (-0.5, +0.3), 14: (-0.5, -0.4),
               15: (+0.1, -0.9), 13: (+0.6, -0.5)},
    "flnm": {2: (+0.9, -0.5), 3: (-0.9, -0.5), 4: (-0.4, +0.6)},
}
D_TITLE = {
    "lesion": "d. Per-cluster mean voxel-level lesion frequency",
    "flnm":   "d. Per-cluster mean voxel-level functional lesion connectivity (t)",
}
# (long_label, cb unit, cmap, cb_min, cb_max, low tick text, high tick text)
VIEW_LABEL = {
    "lesion": ("lesion presence (acute infarct mask)", "%", "hot",
               0.0, 0.238, "0", "24"),
    "flnm":   ("functional lesion connectivity (fLNM t-map)", "t", "coolwarm",
               -23.4, 23.4, "-23", "+23"),
}


def _grid_for(n_clusters):
    if n_clusters == 9:
        return 3, 3
    if n_clusters == 16:
        return 4, 4
    if n_clusters == 12:
        return 3, 4
    if n_clusters <= 6:
        return 2, 3
    n_cols = 3
    return (n_clusters + n_cols - 1) // n_cols, n_cols


def _build_suppl_fig3(view):
    long_label, cb_label, cmap_name, cb_min, cb_max, cb_lo_txt, cb_hi_txt = VIEW_LABEL[view]
    UMAP_CSV_V = EXP / f"umap_{view}_subjects.csv"
    CELLS_DIR = (PROJECT_ROOT / "outputs" / "feature_importance" / "figures"
                 / f"cells_suppl_fig3_{view}_clusters")
    if not UMAP_CSV_V.exists():
        print(f"[skip suppl_fig3_{view}: {UMAP_CSV_V} missing]")
        return None

    df = pd.read_csv(UMAP_CSV_V)
    pred = pd.read_csv(PRED_ALL7)
    pred_psci = pred[pred["target"] == "cognition-PSCI_present"][
        ["subject_id", "y_true", "mean_meta_proba"]].rename(
            columns={"mean_meta_proba": "psci_prob", "y_true": "psci_true"})
    df = df.merge(pred_psci, on="subject_id", how="left")

    clusters = sorted([int(c) for c in df["cluster"].unique() if c >= 0])
    n_clu = len(clusters)
    n_rows_grid, n_cols_grid = _grid_for(n_clu)
    print(f"[suppl_fig3_{view}] {n_clu} clusters -> {n_rows_grid}x{n_cols_grid} grid")

    cmap_cl = plt.cm.tab20 if n_clu > 10 else plt.cm.tab10
    colors = {c: cmap_cl(i % 20) for i, c in enumerate(clusters)}
    colors[-1] = (0.75, 0.75, 0.75, 0.55)

    grid_row_h = 2.4 * n_rows_grid
    fig = plt.figure(figsize=(22, 4 + grid_row_h + 6.5 + 6.5))
    outer = GridSpec(4, 1, figure=fig, hspace=0.10,
                     left=0.04, right=0.985, top=0.98, bottom=0.020,
                     height_ratios=[0.70, grid_row_h / 3.0, 0.75, 0.75])

    # Row 0: three UMAP panels
    sub = GridSpecFromSubplotSpec(1, 3, subplot_spec=outer[0], wspace=0.10)
    ax_a = fig.add_subplot(sub[0, 0])
    noise = df[df["cluster"] == -1]
    ax_a.scatter(noise["umap_x"], noise["umap_y"], s=10, c=[colors[-1]],
                 alpha=0.5, linewidths=0)
    for c in clusters:
        sub_c = df[df["cluster"] == c]
        ax_a.scatter(sub_c["umap_x"], sub_c["umap_y"], s=18, c=[colors[c]],
                     alpha=0.78, linewidths=0)
        cx, cy = sub_c["umap_x"].mean(), sub_c["umap_y"].mean()
        rate = (sub_c["psci_true"].dropna().mean()
                if sub_c["psci_true"].notna().any() else None)
        disp = c + 1
        lab = f"{disp}\n{rate*100:.0f}%" if rate is not None else f"{disp}"
        dx, dy = CLUSTER_LABEL_NUDGE_V.get(view, {}).get(c, (0.0, 0.0))
        ax_a.text(cx + dx, cy + dy, lab, fontsize=10, fontweight="bold",
                  color="black", ha="center", va="center",
                  bbox=dict(boxstyle="round,pad=0.28", fc="white",
                            ec="black", lw=1.0, alpha=0.92), zorder=10)
    ax_a.set_title(f"a. UMAP of {SHORT_LABEL[view]} features",
                   loc="left", fontsize=14, fontweight="bold")
    ax_a.text(0.1, 0.08, "Coloured by HDBSCAN cluster", fontsize=12,
              transform=ax_a.transAxes)
    ax_a.margins(0.10)
    _strip_axes(ax_a)

    ax_b = fig.add_subplot(sub[0, 1])
    has = df["psci_prob"].notna()
    if (~has).any():
        ax_b.scatter(df.loc[~has, "umap_x"], df.loc[~has, "umap_y"], s=10,
                     c=[(0.85, 0.85, 0.85, 0.5)], linewidths=0)
    vmin = float(df.loc[has, "psci_prob"].quantile(0.05))
    vmax = float(df.loc[has, "psci_prob"].quantile(0.95))
    sc = ax_b.scatter(df.loc[has, "umap_x"], df.loc[has, "umap_y"], s=18,
                      c=df.loc[has, "psci_prob"], cmap="plasma",
                      vmin=vmin, vmax=vmax, alpha=0.88, linewidths=0)
    cb = plt.colorbar(sc, ax=ax_b, shrink=0.55, pad=0.01, location="right")
    cb.set_label(f"PSCI probability\n(clip {vmin:.2f}\u2013{vmax:.2f})",
                 fontsize=11, fontweight="bold")
    cb.ax.tick_params(labelsize=10)
    ax_b.set_title("b. Predicted PSCI probability (integrated model)",
                   loc="left", fontsize=14, fontweight="bold")
    ax_b.margins(0.10)
    _strip_axes(ax_b)

    ax_c = fig.add_subplot(sub[0, 2])
    _scatter_binary(ax_c, df, "psci_true", dot_sizes=(10, 16, 18))
    ax_c.set_title("c. Ground-truth PSCI",
                   loc="left", fontsize=14, fontweight="bold", x=-0.025)
    ax_c.legend(handles=_binary_legend_handles()[:2], loc="upper right",
                bbox_to_anchor=(1.0, -0.01), ncol=2,
                fontsize=10, frameon=False, handletextpad=0.4,
                columnspacing=1.6, borderaxespad=0.0)
    ax_c.margins(0.10)
    _strip_axes(ax_c)

    # Row 1: cluster axial-slice block
    sub_d_outer = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[1],
                                          hspace=0.10,
                                          height_ratios=[0.30 / n_rows_grid, 1.0])
    ax_d_title = fig.add_subplot(sub_d_outer[0])
    ax_d_title.set_xticks([]); ax_d_title.set_yticks([])
    for sp in ax_d_title.spines.values():
        sp.set_visible(False)
    ax_d_title.text(0.0, 0.5, D_TITLE[view], ha="left", va="center",
                    fontsize=15, fontweight="bold", transform=ax_d_title.transAxes)
    sub_d = GridSpecFromSubplotSpec(n_rows_grid, n_cols_grid + 1,
                                    subplot_spec=sub_d_outer[1],
                                    wspace=0.02, hspace=0.20,
                                    width_ratios=[1.0] * n_cols_grid + [0.025])
    slice_axes = []
    for i, cid in enumerate(clusters):
        r, c_idx = divmod(i, n_cols_grid)
        ax = fig.add_subplot(sub_d[r, c_idx])
        png = CELLS_DIR / f"cluster_+{cid}_axial.png"
        if png.exists():
            ax.imshow(mpimg.imread(png))
        n_sub = int(df[df["cluster"] == cid].shape[0])
        ax.set_title(f"{cid + 1}  (N = {n_sub})", fontsize=14, fontweight="bold")
        _strip_axes(ax)
        slice_axes.append(ax)
    for j in range(len(clusters), n_rows_grid * n_cols_grid):
        r, c_idx = divmod(j, n_cols_grid)
        ax_empty = fig.add_subplot(sub_d[r, c_idx])
        _strip_axes(ax_empty)
        ax_empty.set_visible(False)
    mid_row = max(0, (n_rows_grid - 1) // 2)
    cbax = fig.add_subplot(sub_d[mid_row, n_cols_grid])
    _pos = cbax.get_position()
    cbax.set_position([_pos.x0 + 0.018, _pos.y0, _pos.width, _pos.height])
    cbax.imshow(np.linspace(0, 1, 256).reshape(-1, 1),
                cmap=cmap_name, aspect="auto", origin="lower")
    cbax.set_xticks([])
    cbax.set_yticks([0, 255])
    cbax.set_yticklabels([cb_lo_txt, cb_hi_txt], fontsize=11, fontweight="bold")
    for sp in cbax.spines.values():
        sp.set_linewidth(0.5)
    cbax.set_title(cb_label, fontsize=12, fontweight="bold", pad=6)

    # Row 2: per-domain predicted prob
    outer_row = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[2],
                                        hspace=0.65, height_ratios=[0.10, 1.0])
    ax_title = fig.add_subplot(outer_row[0])
    ax_title.set_xticks([]); ax_title.set_yticks([])
    for sp in ax_title.spines.values():
        sp.set_visible(False)
    ax_title.text(0.0, 0.5,
                  "e. Predicted probability for each cognitive domain (integrated model)",
                  ha="left", va="center", fontsize=15, fontweight="bold",
                  transform=ax_title.transAxes)
    n = len(DOMAINS_FIG3)
    sub_e = GridSpecFromSubplotSpec(1, n + 1, subplot_spec=outer_row[1],
                                    wspace=0.10, width_ratios=[1.0] * n + [0.06])
    df_um = df[["subject_id", "umap_x", "umap_y"]]
    for i, (tgt, nice) in enumerate(DOMAINS_FIG3):
        ax = fig.add_subplot(sub_e[0, i])
        p_tgt = pred[pred["target"] == tgt][["subject_id", "mean_meta_proba"]]
        merged = df_um.merge(p_tgt, on="subject_id", how="left")
        has = merged["mean_meta_proba"].notna()
        if (~has).any():
            ax.scatter(merged.loc[~has, "umap_x"], merged.loc[~has, "umap_y"],
                       s=6, c=[(0.88, 0.88, 0.88, 0.4)], linewidths=0)
        v_lo = float(merged.loc[has, "mean_meta_proba"].quantile(0.05))
        v_hi = float(merged.loc[has, "mean_meta_proba"].quantile(0.95))
        ax.scatter(merged.loc[has, "umap_x"], merged.loc[has, "umap_y"], s=10,
                   c=merged.loc[has, "mean_meta_proba"], cmap="plasma",
                   vmin=v_lo, vmax=v_hi, alpha=0.85, linewidths=0)
        ax.set_title(f"{nice}\nclip: {v_lo:.2f}\u2013{v_hi:.2f}",
                     fontsize=14, fontweight="bold")
        _strip_axes(ax)
    cbax_e = fig.add_subplot(sub_e[0, n])
    cbax_e.imshow(np.linspace(0, 1, 256).reshape(-1, 1),
                  cmap="plasma", aspect="auto", origin="lower")
    cbax_e.set_xticks([])
    cbax_e.set_yticks([0, 255])
    cbax_e.set_yticklabels(["low", "high"], fontsize=12, fontweight="bold")
    for sp in cbax_e.spines.values():
        sp.set_linewidth(0.5)
    cbax_e.set_title("rel.\nprob.", fontsize=12, fontweight="bold", pad=13)

    # Row 3: per-domain binary
    outer_row = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[3],
                                        hspace=0.65, height_ratios=[0.10, 1.0])
    ax_title = fig.add_subplot(outer_row[0])
    ax_title.set_xticks([]); ax_title.set_yticks([])
    for sp in ax_title.spines.values():
        sp.set_visible(False)
    ax_title.text(0.0, 0.5, "f. Ground-truth per cognitive domain",
                  ha="left", va="center", fontsize=15, fontweight="bold",
                  transform=ax_title.transAxes)
    sub_f = GridSpecFromSubplotSpec(1, n + 1, subplot_spec=outer_row[1],
                                    wspace=0.10, width_ratios=[1.0] * n + [0.06])
    for i, (tgt, nice) in enumerate(DOMAINS_FIG3):
        ax = fig.add_subplot(sub_f[0, i])
        p_tgt = pred[pred["target"] == tgt][["subject_id", "y_true"]]
        merged = df_um.merge(p_tgt, on="subject_id", how="left")
        _scatter_binary(ax, merged, "y_true")
        ax.set_title(f"{nice}", fontsize=14, fontweight="bold")
        _strip_axes(ax)
    leg_ax = fig.add_subplot(sub_f[0, n])
    leg_ax.set_xticks([]); leg_ax.set_yticks([])
    for sp in leg_ax.spines.values():
        sp.set_visible(False)
    leg_ax.legend(handles=_binary_legend_handles(), loc="center left",
                  fontsize=11, frameon=False,
                  handletextpad=0.4, labelspacing=1.1)

    _add_radiological_overlay(fig, slice_axes)

    gray_divider(fig, outer[0].get_position(fig).y0 - 0.016, x0=0.10, x1=0.90)
    gray_divider(fig, outer[1].get_position(fig).y0 - 0.003, x0=0.10, x1=0.90)
    gray_divider(fig, outer[2].get_position(fig).y0 - 0.003, x0=0.10, x1=0.90)

    out_png = OUT_DIR / f"suppl_fig3_{view}_clusters.png"
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
    fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"saved {out_png}")


_build_suppl_fig3("lesion")
_build_suppl_fig3("flnm")

In [ ]:
# Copy standalone supplement figures into manuscript/figures/supplement
MANIFEST = [
    (EXP / 'figures' / 'lesion_distribution_map_analysed.png',
     'suppl_lesion_distribution_map.png',
     'Figure S1 — distribution of acute infarcts.'),
    (CORE / 'pooled_cv_violin_auroc.png',
     'suppl_pooled_cv_violin_auroc.png',
     'Figure S2 — pooled CV performance, AUROC.'),
    (PROJECT_ROOT / 'outputs' / 'predictive_modeling_core' / 'calibration' / 'calibration_curves_pooled.png',
     'suppl_calibration_curves_pooled.png',
     'Figure S3 — calibration of the integrated model.'),
    (CORE / 'modality_comparison_violin.png',
     'suppl_modality_comparison_violin.png',
     'Figure S4 — individual acute infarct representations.'),
    (OUT_DIR / 'suppl_baseline_variable_ablation.png',
     'suppl_baseline_variable_ablation.png',
     'Figure S5 — baseline variable ablation.'),
    (PROJECT_ROOT / 'outputs' / 'predictive_modeling_age_strata' / 'figures' / 'age_strata_delta_strip.png',
     'suppl_age_strata_delta_strip.png',
     'Figure S6 — benefit of the integrated model by age tertile.'),
    (PROJECT_ROOT / 'outputs' / 'predictive_modeling_sex_strata' / 'figures' / 'sex_strata_delta_strip.png',
     'suppl_sex_strata_delta_strip.png',
     'Figure S7 — benefit of the integrated model by sex.'),
    (SENS / 'sensitivity_atlas_resolution.png',
     'suppl_sensitivity_atlas_resolution.png',
     'Figure S8 — parcellation resolution.'),
    (PROJECT_ROOT / 'outputs' / 'predictive_modeling_sensitivity_streamlines' / 'figures' / 'slnm_subsample_sensitivity_violin.png',
     'suppl_slnm_subsample_sensitivity_violin.png',
     'Figure S9 — normative tractogram density.'),
    (PROJECT_ROOT / 'outputs' / 'predictive_modeling_sensitivity' / 'log_volume' / 'figures' / 'logvolume_pooled_violin.png',
     'suppl_logvolume_pooled_violin.png',
     'Figure S10 — log-transformed infarct volume.'),
    (PROJECT_ROOT / 'outputs' / 'predictive_modeling_sensitivity' / 'regional_vs_voxelwise' / 'regional_vs_voxelwise.png',
     'suppl_regional_vs_voxelwise.png',
     'Figure S11 — regional versus voxelwise lesion features.'),
    (OUT_DIR / 'main_fig5_loco.png',
     'suppl_loco_generalisation.png',
     'Figure S12 — cross-cohort generalisation (leave-one-cohort-out).'),
    (CORE / 'loco_cv_panel_auroc.png',
     'suppl_loco_panel_auroc.png',
     'Figure S13 — leave-one-cohort-out AUROC.'),
    (EXP / 'figures' / 'loco_imaging_transfers_all_domains.png',
     'suppl_loco_transfers_all_domains.png',
     'Figure S14 — cross-cohort transfer for individual cognitive domains.'),
    (CORE / 'loco_lesion_representations.png',
     'suppl_loco_lesion_representations.png',
     'Figure S15 — cross-cohort performance of the three lesion representations.'),
    (OUT_DIR / 'suppl_beta_top_regions_heatmap.png',
     'suppl_beta_top_regions_heatmap.png',
     'Figure S16 — overlap of the most strongly weighted structural LNM regions.'),
    (OUT_DIR / 'suppl_ridge_betas_lesion.png',
     'suppl_ridge_betas_lesion.png',
     'Figure S17 — lesion location coefficient profiles.'),
    (OUT_DIR / 'suppl_ridge_betas_flnm.png',
     'suppl_ridge_betas_flnm.png',
     'Figure S18 — functional LNM coefficient profiles.'),
    (EXP / 'figures' / 'umap_baseline_overlay.png',
     'suppl_umap_baseline_overlay.png',
     'Figure S19 — baseline covariates on the structural LNM UMAP embedding.'),
    (OUT_DIR / 'suppl_fig3_lesion_clusters.png',
     'suppl_fig3_lesion_clusters.png',
     'Figure S20 — patient stratification by lesion location.'),
    (OUT_DIR / 'suppl_fig3_flnm_clusters.png',
     'suppl_fig3_flnm_clusters.png',
     'Figure S21 — patient stratification by functional LNM.'),
    (PROJECT_ROOT / 'outputs' / 'predictive_modeling_branches' / 'figures' / 'preexisting_absolute_performance.png',
     'suppl_preexisting_absolute_performance.png',
     'Figure S22 — absolute performance with pre-existing lesion features.'),
]

lines = ['# Supplement figures\n', '\n',
         'Auto-populated by the figure composition notebook (analysis/notebooks/19).\n',
         '\n', '| File | Description |\n', '|---|---|\n']
for src, dst_name, desc in MANIFEST:
    dst = SUPP_DIR / dst_name
    if Path(src).exists():
        shutil.copy2(src, dst)
        for ext in ('.pdf', '.svg'):
            alt = Path(src).with_suffix(ext)
            if alt.exists():
                shutil.copy2(alt, dst.with_suffix(ext))
        print(f'OK   {dst_name}')
    else:
        print(f'MISS {dst_name}  ({src})')
    lines.append(f'| `{dst_name}` | {desc} |\n')

(SUPP_DIR / 'README.md').write_text(''.join(lines))
print(f'supplement README at {SUPP_DIR / "README.md"}')